# Session 7 — From testing records to a SIR model

We will use one simple epidemic model to connect individual test records with population-level dynamics. The notebook moves in three steps: people move between SIR states; equations describe those movements; and weekly testing records help estimate the model’s parameters.

The aim is to understand how a model represents social dynamics. This is an introduction, not a full epidemiology course. In the next part of the lesson, we can use the fitted SIR behavior as a reference for an agent-based model.

## 1. A contagious setting as state transitions

Think of each person as being in one of three states at a given time:

| State | Meaning in this model | At time zero |
|---|---|---:|
| **Susceptible (S)** | Can become infected. | Generally almost all the population |
| **Infectious (I)** | Has the infection and can pass it to others. | Generally very few cases |
| **Recovered (R)** | Is no longer infectious and is treated as immune in this model. | Generally zero |

People can move from **S to I** through contact with someone infectious, then from **I to R** as they recover. The model does not include death, births, or reinfection. Its population is closed, so the three groups always add up to (N):

$$
S + I + R = N.
$$

Here, “time zero” means the beginning of the model. These descriptions show a general starting pattern.

## 2. The mathematical representation

The differential equations describe how quickly each group changes over time. Read **dS/dt**, for example, as “how quickly the susceptible group is changing.”

| Differential equation | How to read it |
|---|---|
| $$\frac{dS}{dt}=-\beta\frac{SI}{N}$$ | As time passes (**dt**), the susceptible group gets smaller as infections move people from **S** to **I**. |
| $$\frac{dI}{dt}=\beta\frac{SI}{N}-\gamma I$$ | As time passes (**dt**), the infectious group grows through new infections and shrinks through recoveries. Its change is the balance of those two flows. |
| $$\frac{dR}{dt}=\gamma I$$ | As time passes (**dt**), the recovered group grows as infectious people recover. |

Here, **N** is the total population. The expression **SI/N** represents opportunities for contact between susceptible and infectious people in a well-mixed population. The parameters **β** and **γ** determine the rates of the two flows. Time is measured in **weeks** to match the testing summaries.

### What the parameters tell us

- **$\beta$, transmission rate:** how quickly contact with infectious people creates new infections when almost everyone is susceptible.
- **$\gamma$, recovery rate:** how quickly infectious people move to the recovered state. In this model, $1/\gamma$ is the average time spent infectious.
- **$R_0=\beta/\gamma$, basic reproduction number:** the model’s average number of new infections caused by one infectious person at the start, when nearly everyone is susceptible.

Under these assumptions, $R_0>1$ means infections initially grow; $R_0<1$ means they initially shrink. It is a summary of the model’s starting conditions, not a count of exactly how many people a particular person infects and not, by itself, an official epidemic classification.

Next we will estimate $\beta$ and $\gamma$ from the synthetic records. The estimates will describe this dataset under the SIR assumptions.

If you want to explore the curves with sliders before looking at the records, use the optional [SIR Interactive Playground](https://colab.research.google.com/drive/19YqgS1uoCrNgtyyM3OJuUYm2x9XyONxi?authuser=2). This notebook proceeds directly to estimating the parameters from the testing data.

## 3. From test records to SIR parameters

The CSV contains one row for each completed test. It follows this **synthetic testing policy**:

- The population is **10,000**; this denominator is set here, not repeated in the CSV.
- Week 1 is a random baseline sample of 100 tests, taken before the outbreak is seeded.
- From week 2 onward, there are 80 fresh random surveillance tests each week.
- A person with an earlier positive surveillance test may be scheduled for a follow-up **four weeks later**. The four-week wait is a choice in this teaching scenario; it is not inferred from the data or asserted as the pathogen’s infectious duration. Up to 20 follow-up kits are available each week. If more than 20 follow-ups are due, 20 are selected randomly. A person who does not return leaves a kit unused, and the sample is not backfilled.

We use the random **surveillance** sample to estimate the share infectious. Follow-ups are targeted at earlier positive cases, so we use them separately to estimate recovery.

### Step 1: open the CSV

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import binom

POPULATION = 10_000
TESTS_URL = "https://raw.githubusercontent.com/DACSS-Complexity/s7_epidemics/main/session7_synthetic_tests.csv"

records = pd.read_csv(TESTS_URL, parse_dates=["date"])
records.head()

### Step 2: keep the surveillance sample

A surveillance sample is meant to represent the population. A follow-up sample is different: it deliberately retests people who tested positive earlier. Combining the two would make the positivity rate look higher than the population sample suggests.

First, check how many records belong to each test type.

In [ ]:
records.test_type.value_counts()

Now keep only the surveillance tests. The `positive` column converts the result into a number: **1** for a positive test and **0** for a negative test. That lets us add the column to count positives in each week.

In [ ]:
surveillance = records.loc[records.test_type.eq("surveillance")].copy()
surveillance["positive"] = surveillance.result.eq("positive").astype(int)

### Step 3: summarize tests by week

For each week, count how many surveillance tests were completed and how many were positive. Keeping both counts matters: 10 positives out of 20 tests tells a different sampling story from 10 out of 80.

In [ ]:
weekly = surveillance.groupby("week").agg(
    date=("date", "min"),
    tests=("test_id", "size"),
    positives=("positive", "sum"),
)

weekly.head(8)

### Step 4: turn positivity into an SIR observation

The **positivity rate** is the share of surveillance tests that are positive. If the sample is random and the tests are accurate, it gives a sample-based estimate of the infectious share, $I/N$. Multiplying by the population gives an approximate infectious count; it is not a census of all 10,000 people.

In [ ]:
weekly["positivity"] = weekly.positives / weekly.tests

weekly[["tests", "positives", "positivity"]].round(3)

The rising positivity in this synthetic dataset gives us a pattern to compare with the SIR model. Week 1 is the pre-outbreak baseline, so we use weeks 2 onward when fitting the outbreak trajectory. A zero in a small weekly sample would mean “no positives observed in this sample,” not “no infectious people exist.”

In [ ]:
ax = weekly.positivity.mul(100).plot(marker="o", color="#087e8b")
ax.set(title="Weekly surveillance positivity", xlabel="Week", ylabel="Positive tests (%)")
ax.grid(alpha=0.2)
plt.show()

### Step 5: use follow-up tests to estimate $\gamma$

We will use the follow-up data in three steps:

- **Use the testing schedule:** eligible people are scheduled for a retest four weeks after a positive surveillance test. This interval is set by the synthetic testing policy; it is not a fixed infectious duration.

In [ ]:
FOLLOWUP_WEEKS = 4

# Find each person's first positive surveillance test.
first_positive = records.loc[
    records.test_type.eq("surveillance") & records.result.eq("positive"),
    ["person_id", "week"],
].sort_values("week").drop_duplicates("person_id")
first_positive = first_positive.rename(columns={"week": "positive_week"})

# Match follow-up tests that occur four weeks later.
followups = records.loc[records.test_type.eq("followup")].rename(
    columns={"week": "followup_week"}
)
returned = followups.merge(first_positive, on="person_id", how="inner")
returned = returned.loc[
    returned.followup_week - returned.positive_week == FOLLOWUP_WEEKS
].copy()

returned[["person_id", "positive_week", "followup_week", "result"]].head()

- **Interpret the retest:** a positive result means the infection is still detected; for this exercise, a negative result is treated as recovered by the retest. The test shows status at that later point, not the exact week of recovery. People without a completed follow-up are not counted. First, count the completed follow-up results:

In [8]:
returned.result.value_counts()

- **Estimate the recovery rate:** assume a constant recovery rate. Then the chance of still being infectious after $d$ weeks is $e^{-\gamma d}$. The share still positive at the four-week retest informs $\gamma$. The model-implied mean infectious duration is $1/\gamma$ weeks; individual durations are not all fixed at that value.<sup>1</sup>

<sup>1</sup> **Connection to the reading:** Weiss (2013, Section 2.1) uses $\nu$ for the SIR recovery rate and relates infectious duration to $D=1/\nu$. Here, $\gamma$ plays the role of $\nu$. The exponential survival expression above follows from the notebook’s constant-recovery-rate assumption; it is not stated directly in the reading.

In [ ]:
still_positive_share = returned.result.eq("positive").mean()
GAMMA_ESTIMATE = -np.log(still_positive_share) / FOLLOWUP_WEEKS
AVERAGE_INFECTIOUS_WEEKS = 1 / GAMMA_ESTIMATE

print(f"Four-week follow-ups used: {len(returned)}")
print(f"Estimated gamma: {GAMMA_ESTIMATE:.3f} per week")
print(f"Model-implied mean infectious duration: {AVERAGE_INFECTIOUS_WEEKS:.1f} weeks (not fixed for each person)")

This estimate depends on the synthetic four-week follow-up schedule, the assumption of a constant recovery rate, and treating a negative retest as recovered. In real data, a negative test is evidence from a test, not proof of exactly when someone stopped being infectious.

### Step 6: prepare the surveillance counts for fitting $\beta$

We will compare the SIR model’s expected infectious share with the observed number of positives out of the tested sample each week. **For this synthetic dataset, fitting starts in week 2** because week 1 is the pre-outbreak baseline. With another dataset, the fit could start later if early weeks are incomplete, unreliable, or before the outbreak period of interest; state the reason for the chosen start week. The fitting procedure estimates the infectious count at the start of the fitted period from the surveillance data. For simplicity, it assumes $R=0$ at that point.

In [ ]:
observed = weekly.loc[weekly.index >= 2]
observed_weeks = observed.index.to_numpy(dtype=int)
observed_n = observed.tests.to_numpy(dtype=int)
observed_positive = observed.positives.to_numpy(dtype=int)

print(f"Weeks used in fit: {len(observed)} (weeks {observed_weeks[0]}–{observed_weeks[-1]})")

### Step 7: describe one week of SIR movement

The differential equations describe continuous change. To connect them to weekly observations, the function below advances the compartments one week at a time. It uses the probability of infection or recovery over a week, based on $\beta$ and $\gamma$.

In [ ]:
def sir_one_week(susceptible, infectious, recovered, beta, gamma):
    new_infections = susceptible * (-np.expm1(-beta * infectious / POPULATION))
    new_recoveries = infectious * (-np.expm1(-gamma))

    susceptible -= new_infections
    infectious += new_infections - new_recoveries
    recovered += new_recoveries
    return susceptible, infectious, recovered

Now apply that one-week movement repeatedly to get the model’s expected S, I, and R for the observed weeks.

In [ ]:
def sir_states_by_week(beta, initial_infectious):
    susceptible = POPULATION - initial_infectious
    infectious = initial_infectious
    recovered = 0.0
    states = []
    previous_week = int(observed_weeks[0])

    for week in observed_weeks:
        for _ in range(int(week - previous_week)):
            susceptible, infectious, recovered = sir_one_week(
                susceptible, infectious, recovered, beta, GAMMA_ESTIMATE
            )
        states.append((susceptible, infectious, recovered))
        previous_week = int(week)

    return np.asarray(states)

### Step 8: let the data estimate $\beta$

The model predicts a probability of testing positive in each weekly surveillance sample: its infectious share, $I/N$. We score how well those probabilities match the observed positive and negative counts, then let an optimizer search for the $\beta$ and starting infectious count that best fit the data.

This is a statistical fit, not a direct reading of $\beta$ from any single CSV column.

In [ ]:
def negative_log_likelihood(log_parameters):
    beta, initial_infectious = np.exp(log_parameters)
    states = sir_states_by_week(beta, initial_infectious)
    model_positive_share = np.clip(states[:, 1] / POPULATION, 1e-12, 1 - 1e-12)

    # Each week's positives are counted out of that week's surveillance tests.
    return -binom.logpmf(observed_positive, observed_n, model_positive_share).sum()

The function above assigns a mismatch score to each candidate $\beta$ and starting infectious count. A smaller score means the model's predicted positive-test counts are closer to the observed counts. In the next cell, the optimizer searches for the parameter pair with the smallest score. The returned values are converted back from log scale, and then $R_0$ is calculated using the fitted $\beta$ and $\gamma$.

In [ ]:
fit = minimize(
    negative_log_likelihood,
    x0=np.log([0.5, 20]),
    method="L-BFGS-B",
    bounds=[(np.log(0.001), np.log(2.0)),
            (np.log(0.1), np.log(POPULATION - 1))],
)

BETA_ESTIMATE, INITIAL_INFECTIOUS_ESTIMATE = np.exp(fit.x)
R0_ESTIMATE = BETA_ESTIMATE / GAMMA_ESTIMATE

print(f"Estimated beta: {BETA_ESTIMATE:.3f} per week")
print(f"Estimated gamma: {GAMMA_ESTIMATE:.3f} per week")
print(f"Estimated R0 = beta / gamma: {R0_ESTIMATE:.2f}")
print(f"Estimated infectious people at start of week 2: {INITIAL_INFECTIOUS_ESTIMATE:.0f}")

### Step 9: compare the fitted SIR curve with the sample

The dots show the weekly share of surveillance tests that were positive. The line shows the infectious share produced by the fitted SIR model. The model is trying to follow the overall pattern; it does not need to pass through every sample point.

In [ ]:
fitted_states = sir_states_by_week(BETA_ESTIMATE, INITIAL_INFECTIOUS_ESTIMATE)

plt.scatter(observed_weeks, 100 * observed_positive / observed_n,
            color="#a7bdc5", label="Observed surveillance positivity")
plt.plot(observed_weeks, 100 * fitted_states[:, 1] / POPULATION,
         color="#087e8b", linewidth=2, label="Fitted SIR infectious share")
plt.xlabel("Week")
plt.ylabel("Share (%)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### What we learned from the fit

The testing sample gives evidence about how the infectious share changes over time. The follow-up sample gives evidence about how quickly detected infections move toward recovery under the model’s assumptions. Together, they let us estimate $\beta$, $\gamma$, and the implied $R_0$.

These are **population-level, model-based estimates**. The SIR model has no contact tracing information, so it cannot tell us who infected whom or identify high-contact individuals. The estimates give us a data-informed reference for the next step: parameterizing a simple ABM and then exploring clearly stated social scenarios.

For Homework 3, use the separate parameter-interval notebook in the `Homework_3` repository to extend this analysis by estimating sampling intervals for the fitted rates.

### Assumptions to keep in mind

This teaching fit assumes random surveillance sampling, accurate tests, a closed population, homogeneous mixing, lasting immunity, and a constant recovery rate. The four-week retest interval is set by the synthetic testing policy; it is not an assumed fixed infectious duration. A negative follow-up is treated as recovered for this exercise. The SIR model summarizes population behavior; it does not identify individual contact patterns. The estimates are conditional on these assumptions and on this small synthetic sample.

## Why continue with an ABM?

The SIR model uses the surveillance and follow-up data to estimate population-level transmission and recovery rates ($\beta$ and $\gamma$), and the implied $R_0$. It gives us a data-informed reference for the outbreak, but it assumes homogeneous mixing: it does not represent where people meet or how a policy changes those encounters.

The class ABM carries the estimated $\beta$ and $\gamma$ into an agent-based setting. Agents move among patches, and exposure can occur when susceptible and infectious agents share a patch. We can then represent two policy options from the case: some agents wear masks, reducing transmission according to an assumed mask effect; some stay where they are, changing how they move and encounter others. The model lets us compare these policies separately and together, while varying their coverage independently.

For each scenario, the ABM tracks the height of the infectious peak and the week when it occurs. BehaviorSpace can compare those outcomes across policy combinations and the $\beta$ and $\gamma$ intervals estimated in Notebook 2. This shows how the modeled policies behave under the assumptions we specified; it does not establish their real-world effects or predict exactly what will happen. The testing data estimate the SIR rates, while the mask effect and the way movement shapes encounters are additional modeling assumptions.